In [4]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

In [2]:
#biblioteke
import pandas as pd

from dotenv import load_dotenv
from google import genai

In [ ]:
#uvoz funkcije za generiranje sintetickih podataka
from src.generation import (generate_synthetic_dataset)

In [5]:
#putanje do podataka
PROCESSED_DIR = Path("data/processed")
TRAIN_PATH = PROCESSED_DIR / "train.csv"
SYNTHETIC_PATH = PROCESSED_DIR / "synthetic_train.csv"
AUGMENTED_PATH = PROCESSED_DIR / "augmented_train.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"
CHALLENGE_PATH = PROCESSED_DIR / "challenge_test.csv"

In [14]:
#ucitavanje train podataka
train_df = pd.read_csv(TRAIN_PATH)

print("Broj train primjera:",len(train_df))
train_df.head()

Broj train primjera: 31268


,title,text,full_text,label,original_type,source_id
0,Philippine troops kill 14 Maoist rebels in cla...,MANILA ( ) - Philippine soldiers have killed 1...,Philippine troops kill 14 Maoist rebels in cla...,0,real,isot_13416
1,EU border controls could be extended in crisis...,BRUSSELS ( ) - Temporary border controls insid...,EU border controls could be extended in crisis...,0,real,isot_18702
2,Trump Whines About Hillary In Response To Obst...,After the news broke that Trump is going to be...,Trump Whines About Hillary In Response To Obst...,1,fake,isot_22321
3,Trump Throws PETTY Hissy Fit Because Bill Clin...,"Once again, Donald Trump is totally predictabl...",Trump Throws PETTY Hissy Fit Because Bill Clin...,1,fake,isot_24529
4,John Kerry commits more U.S. military aid for ...,TBILISI ( ) - Secretary of State John Kerry to...,John Kerry commits more U.S. military aid for ...,0,real,isot_08809


In [15]:
#izdvajanje stvarnih vijesti iz train skupa
real_train = (
    train_df[train_df["label"] == 0]
    .copy()
    .reset_index(drop=True)
)
print("Broj stvarnih vijesti:", len(real_train))
real_train.head()

Broj stvarnih vijesti: 16955


,title,text,full_text,label,original_type,source_id
0,Philippine troops kill 14 Maoist rebels in cla...,MANILA ( ) - Philippine soldiers have killed 1...,Philippine troops kill 14 Maoist rebels in cla...,0,real,isot_13416
1,EU border controls could be extended in crisis...,BRUSSELS ( ) - Temporary border controls insid...,EU border controls could be extended in crisis...,0,real,isot_18702
2,John Kerry commits more U.S. military aid for ...,TBILISI ( ) - Secretary of State John Kerry to...,John Kerry commits more U.S. military aid for ...,0,real,isot_08809
3,Germany lauds anti-nuclear campaign winning No...,BERLIN ( ) - Germany on Friday welcomed the No...,Germany lauds anti-nuclear campaign winning No...,0,real,isot_17983
4,"In Cuba visit, Colorado governor sees governme...",HAVANA ( ) - The governor of Colorado said on ...,"In Cuba visit, Colorado governor sees governme...",0,real,isot_05598


In [16]:
# ucitavanje Gemini API kljuca

load_dotenv(".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY nije pronaden")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

GENERATION_MODEL = "gemini-3.5-flash-lite"

In [17]:
N_GENERATION_SOURCES = 1000

#odabir stvarnih train vrijednosti koje ce se koristiti za generiranje
generation_sources = (
    real_train
    .sample(
        n=min(N_GENERATION_SOURCES, len(real_train)),
        random_state=42
    )
    .reset_index(drop=True)
)

print("Broj originalnih izvora:", len(generation_sources))
print("Maksimalan broj synthetic primjera:", len(generation_sources) * 3)

Broj originalnih izvora: 1000
Maksimalan broj synthetic primjera: 3000


In [18]:
#generiranje synthetic train skupa
synthetic_train = generate_synthetic_dataset(
        source_df=generation_sources,
        client=client,
        model_name=GENERATION_MODEL
)
print("Ukupno synthetic primjera:", len(synthetic_train))

Generiranje 1/1000
Generiranje 2/1000
Generiranje 3/1000
Generiranje 4/1000
Generiranje 5/1000
Generiranje 6/1000
Generiranje 7/1000
Generiranje 8/1000
Generiranje 9/1000
Generiranje 10/1000
Generiranje 11/1000
Generiranje 12/1000
Generiranje 13/1000
Generiranje 14/1000
Generiranje 15/1000
Generiranje 16/1000
Generiranje 17/1000
Generiranje 18/1000
Generiranje 19/1000
Generiranje 20/1000
Generiranje 21/1000
Generiranje 22/1000
Generiranje 23/1000
Generiranje 24/1000
Generiranje 25/1000
Generiranje 26/1000
Generiranje 27/1000
Generiranje 28/1000
Generiranje 29/1000
Generiranje 30/1000
Generiranje 31/1000
Generiranje 32/1000
Generiranje 33/1000
Generiranje 34/1000
Generiranje 35/1000
Generiranje 36/1000
Generiranje 37/1000
Generiranje 38/1000
Generiranje 39/1000
Generiranje 40/1000
Generiranje 41/1000
Generiranje 42/1000
Generiranje 43/1000
Generiranje 44/1000
Generiranje 45/1000
Generiranje 46/1000
Generiranje 47/1000
Generiranje 48/1000
Generiranje 49/1000
Generiranje 50/1000
Generiran

In [19]:
#ucitavanje postojeceg synthetic_train.csv (ne pokretati ponovno ono gore)
synthetic_train = pd.read_csv(
    SYNTHETIC_PATH
)

print(
    "Ucitanih synthetic primjera:",
    len(synthetic_train)
)

synthetic_train.head()

Ucitanih synthetic primjera: 2997


,source_id,title,text,full_text,label,original_type,manipulation_type,change_summary,synthetic
0,isot_16792,Soccer star Weah to face vice president in Lib...,MONROVIA ( ) - Former soccer star George Weah ...,Soccer star Weah to face vice president in Lib...,1,synthetic,fact_change,Changed the vote margin lead from 10 points to...,True
1,isot_16792,You Won't Believe What This Soccer Legend Just...,MONROVIA ( ) - Former soccer star George Weah ...,You Won't Believe What This Soccer Legend Just...,1,synthetic,clickbait,"Rewrote the title into a dramatic, attention-g...",True
2,isot_16792,Shocking Showdown: Soccer Icon Weah Collides W...,MONROVIA ( ) - In a stunning political earthqu...,Shocking Showdown: Soccer Icon Weah Collides W...,1,synthetic,tone_shift,Shifted the tone of the entire article to be d...,True
3,isot_13888,Russia sends research ship to help search for ...,MOSCOW ( ) - Russia s defense ministry has sen...,Russia sends research ship to help search for ...,1,synthetic,fact_change,Changed the depth capability of the Yantar fro...,True
4,isot_13888,You Won't Believe Why Russia Just Sent This Ma...,MOSCOW ( ) - Russia s defense ministry has sen...,You Won't Believe Why Russia Just Sent This Ma...,1,synthetic,clickbait,Rewrote the title into a dramatic clickbait st...,True


In [20]:
# uklanjanje duplikata ako postoje
synthetic_train = (
    synthetic_train
    .drop_duplicates(subset=["full_text"])
    .reset_index(drop=True)
)

In [ ]:
# spremanje synthetic skupa (nema potrebe pokretati opet)
synthetic_train.to_csv(
    SYNTHETIC_PATH,
    index=False
)

In [21]:
#priprema originalnih train podataka
original_train = train_df.copy()

original_train["manipulation_type"] = "original"
original_train["change_summary"] = ""
original_train["synthetic"] = False
original_train["data_source"] = "original_train"
synthetic_train["data_source"] = "synthetic"

In [22]:
# isti stupci za originalne i sinteticke podatke
columns = [
    "source_id",
    "title",
    "text",
    "full_text",
    "label",
    "original_type",
    "manipulation_type",
    "change_summary",
    "synthetic",
    "data_source"
]

In [23]:
original_train = original_train[columns]
synthetic_train = synthetic_train[columns]

In [24]:
# spajanje originalnih i sintetickih podataka
augmented_train = pd.concat(
    [original_train, synthetic_train],
    ignore_index=True
)

print("Original train:", len(original_train))
print("Synthetic:", len(synthetic_train))
print("Augmented:", len(augmented_train))

Original train: 31268
Synthetic: 2997
Augmented: 34265


In [ ]:
# spremanje augmented train skupa (ne dirat ponovno)
augmented_train.to_csv(
    AUGMENTED_PATH,
    index=False
)

print("Augmented train podaci spremljeni")

Podaci spremljeni.


In [25]:
#ucitavanje testnog skupa
test_df = pd.read_csv(
    TEST_PATH
)

print(
    "Broj test primjera:",
    len(test_df)
)

Broj test primjera: 7818


In [26]:
#izdvajanje samo stvarnih vijesti iz test skupa (za generiranje challenge testa (tj fake vijesti uz pomoc llm-a))
#fake vijesti iz testnog dijela odbacujemo, te ce u challenge_test.csv bit samo prave vijesti te generirane fake vijesti 

real_test = (
    test_df[
        test_df["label"] == 0
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Broj real test vijesti:",
    len(real_test)
)

Broj real test vijesti: 4239


In [27]:
#odabir stvarnih test vijesti za challenge test
SEED = 42

N_CHALLENGE_SOURCES = 200

challenge_sources = (
    real_test
    .sample(
        n=min(
            N_CHALLENGE_SOURCES,
            len(real_test)
        ),
        random_state=SEED
    )
    .reset_index(drop=True)
)

print(
    "Broj challenge izvora:",
    len(challenge_sources)
)

Broj challenge izvora: 200


In [28]:
#za GENERATION_MODEL koristio se gemini-3.5.-flash-lite tako augmented model nece nauciti specifican stil generatora kojeg je vidio tokom treninga
CHALLENGE_MODEL = "gemini-3.5-flash"

In [ ]:
#generiranje challenge testa (tj generiranje fake vijesti iz stvarnih vijesti iz testnog skupa) (ne pokretati ponovno)
challenge_synthetic = (
    generate_synthetic_dataset(
        source_df=challenge_sources,
        client=client,
        model_name=CHALLENGE_MODEL
    )
)

print(
    "Generirano challenge primjera:",
    len(challenge_synthetic)
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Generiranje 1/200
Generiranje 2/200
Generiranje 3/200
Generiranje 4/200
Generiranje 5/200
Generiranje 6/200
Generiranje 7/200
Generiranje 8/200
Generiranje 9/200
Generiranje 10/200
Generiranje 11/200
Generiranje 12/200
Generiranje 13/200
Generiranje 14/200
Generiranje 15/200
Generiranje 16/200
Generiranje 17/200
Generiranje 18/200
Generiranje 19/200
Generiranje 20/200
Generiranje 21/200
Generiranje 22/200
Generiranje 23/200
Generiranje 24/200
Generiranje 25/200
Generiranje 26/200
Generiranje 27/200
Generiranje 28/200
Generiranje 29/200
Generiranje 30/200
Generiranje 31/200
Generiranje 32/200
Generiranje 33/200
Generiranje 34/200
Generiranje 35/200
Generiranje 36/200
Generiranje 37/200
Generiranje 38/200
Generiranje 39/200
Generiranje 40/200
Generiranje 41/200
Generiranje 42/200
Generiranje 43/200
Generiranje 44/200
Generiranje 45/200
Generiranje 46/200
Generiranje 47/200
Generiranje 48/200
Generiranje 49/200
Generiranje 50/200
Generiranje 51/200
Generiranje 52/200
Generiranje 53/200
Ge

In [ ]:
#ne pokretati ponovno niti ono sve ispod...samo zadnju celiju za ucitavanje kreiranog challenge_testa
print(
    challenge_synthetic[
        "manipulation_type"
    ]
    .value_counts()
)

manipulation_type
fact_change    200
clickbait      200
tone_shift     200
Name: count, dtype: int64


In [19]:
challenge_synthetic[
    "generator_model"
] = CHALLENGE_MODEL

In [20]:
challenge_synthetic[
    "synthetic"
] = True

In [ ]:
#oznacavanje originalnih test vijesti za challenge test
challenge_real = (
    challenge_sources.copy()
)

challenge_real[
    "manipulation_type"
] = "original"

challenge_real[
    "change_summary"
] = ""

challenge_real[
    "synthetic"
] = False

challenge_real[
    "generator_model"
] = "none"

In [22]:
challenge_columns = [
    "source_id",
    "title",
    "text",
    "full_text",
    "label",
    "original_type",
    "manipulation_type",
    "change_summary",
    "synthetic",
    "generator_model"
]

In [ ]:
challenge_real = challenge_real[challenge_columns]

challenge_synthetic = challenge_synthetic[challenge_columns]

In [ ]:
#spajanje originalnih i sintetickih podataka u challenge test
challenge_test = pd.concat(
    [
        challenge_real,
        challenge_synthetic
    ],
    ignore_index=True
)

In [25]:
print(
    "Original real:",
    len(challenge_real)
)

print(
    "Synthetic manipulated:",
    len(challenge_synthetic)
)

print(
    "Challenge ukupno:",
    len(challenge_test)
)

Original real: 200
Synthetic manipulated: 600
Challenge ukupno: 800


In [ ]:
#spremanje
challenge_test.to_csv(
    CHALLENGE_PATH,
    index=False
)

print(
    "Challenge test spremljen:"
)

print(
    CHALLENGE_PATH
)

Challenge test spremljen:
c:\Users\Korisnik\Desktop\ZNANSTVENO_PROGRAMIRANJE\projekt\2025-sci-prog\projects\synteticnewswithLLMs-ldanolic\data\processed\challenge_test.csv


In [29]:
#ucitavanje generiranog dataseta
challenge_test = pd.read_csv(
    CHALLENGE_PATH
)

print(
    "Broj spremljenih challenge primjera:",
    len(challenge_test)
)

print(
    challenge_test[
        "manipulation_type"
    ].value_counts()
)

challenge_test.head()

Broj spremljenih challenge primjera: 800
manipulation_type
original       200
fact_change    200
clickbait      200
tone_shift     200
Name: count, dtype: int64


,source_id,title,text,full_text,label,original_type,manipulation_type,change_summary,synthetic,generator_model
0,isot_15152,Cuban opposition falls at first hurdle as Cast...,HAVANA ( ) - Cuban opposition leaders said the...,Cuban opposition falls at first hurdle as Cast...,0,real,original,NaN,False,none
1,isot_06042,Ryan says Trump to address joint session of Co...,WASHINGTON ( ) - U.S. House of Representatives...,Ryan says Trump to address joint session of Co...,0,real,original,NaN,False,none
2,isot_04940,Senators want food safety review when U.S. fir...,WASHINGTON ( ) - Two U.S. senators from Midwes...,Senators want food safety review when U.S. fir...,0,real,original,NaN,False,none
3,isot_10625,House Republicans ready legal fight against Ob...,WASHINGTON ( ) - Republicans in the House of R...,House Republicans ready legal fight against Ob...,0,real,original,NaN,False,none
4,isot_04809,New York governor unveils South Bronx highway ...,( ) - New York Democratic Governor Andrew Cuom...,New York governor unveils South Bronx highway ...,0,real,original,NaN,False,none


In [7]:
train_df = pd.read_csv("data/processed/train.csv")
synthetic_train = pd.read_csv("data/processed/synthetic_train.csv")

augmented_train = pd.concat(
    [train_df, synthetic_train],
    ignore_index=True
)

augmented_train.to_csv(
    "data/processed/augmented_train.csv",
    index=False
)

print("Train:", len(train_df))
print("Synthetic:", len(synthetic_train))
print("Augmented:", len(augmented_train))

Train: 31268
Synthetic: 2997
Augmented: 34265
